<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/05_self_supervised_learning_and_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 1 · Part 4 — Self-supervised pretraining & adapting foundation models
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| 03 | Vision Foundation Models: the CNN era |
| 04 | Vision Transformers |
| **→ 05** | **Self-supervised pretraining & transfer** |

**By the end of this notebook you can**
1. explain how **MAE** and **DINO** pretrain a ViT **without any labels**,
2. use the dense features of **DINOv2** to segment boats with zero training,
3. **benchmark** frozen backbones with linear probing and k-NN,
4. choose between a linear head, a bigger head, partial and full **fine-tuning**.

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
> This is a **show-and-tell** notebook for the lecture: run it top to bottom, there are no exercises. 🎤 marks prompts for the presenter: questions for the audience (click to reveal the answer) or a value to change live. This notebook downloads ~600 MB of pretrained weights.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, re, base64, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    def shrink(m):  # re-encode every frame as a 256-colour PNG: 3-4x smaller, looks the same
        img = Image.open(io.BytesIO(base64.b64decode(m.group(1).replace("\\\n", ""))))
        buf = io.BytesIO(); img.convert("RGB").quantize(256).save(buf, "png", optimize=True)
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    return HTML(re.sub(r"data:image/png;base64,([A-Za-z0-9+/=\\\n]+)", shrink, anim.to_jshtml()))

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

def to_tensor(img):
    """PIL → ImageNet-normalised tensor (3, H, W)."""
    return (torch.tensor(np.asarray(img), dtype=torch.float32).permute(2, 0, 1) / 255 - IMAGENET_MEAN) / IMAGENET_STD

def to_image(x):
    """ImageNet-normalised tensor (3, H, W) → displayable array (H, W, 3)."""
    return (x.cpu() * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

---
# 1 · Why self-supervised?
The ViT needs **a lot** of data (notebook 04). But labels are expensive: ImageNet took years of crowd-sourcing, and in the
maritime domain annotations often need experts (vessel types, sonar imagery, underwater species, …).
Unlabelled images, on the other hand, are cheap: every camera on every ship produces them all day.

**Self-supervised learning (SSL)** invents a *pretext task* whose "labels" come from the data itself:

| Method | Pretext task | Family |
|---|---|---|
| **MAE** (He et al., 2021, Meta) | hide 75 % of the patches, reconstruct the pixels | masked image modelling |
| **DINO** (Caron et al., 2021, Meta) | different **views** (random crops) of the same image must give the same output | self-distillation |
| **DINOv2** (Oquab et al., 2023, Meta) | DINO + masked modelling, at scale (142 M curated images) | both |

---
# 2 · Masked Autoencoders (MAE)
**How it works**
1. divide the image into patches (as in every ViT),
2. **mask 75 %** of the patches at random,
3. the **encoder** processes only the visible 25 % (cheap!),
4. a light **decoder** reconstructs the pixel values of the masked patches,
5. **loss:** MSE on the masked patches only.

**Why 75 %?** Images are highly redundant: with only a few patches missing, the model could just interpolate from neighbours.
Hiding most of the image forces it to *understand the scene*.

In [ ]:
# @title 🎞️ Animation: how much of the image can we hide? { display-mode: "form" }
img = load_image("trawler", size=224)
arr = np.asarray(img) / 255
order = np.random.default_rng(0).permutation(196)          # random masking order of the 14×14 patches
fig, ax = plt.subplots(figsize=(5, 5.3)); ax.axis("off"); im = ax.imshow(arr); ttl = ax.set_title("")
ratios = list(np.arange(0, 0.91, 0.05)) + [0.9] * 4 + [0.75] * 6
def update(f):
    r = ratios[f]; masked = arr.copy()
    for k in order[:int(round(r * 196))]:
        i, j = divmod(k, 14); masked[i * 16:(i + 1) * 16, j * 16:(j + 1) * 16] = 0.5
    im.set_data(masked); ttl.set_text(f"mask ratio {r:.0%}" + ("  ← MAE default" if abs(r - .75) < 1e-6 and f >= len(ratios) - 6 else ""))
    return [im]
play(animation.FuncAnimation(fig, update, frames=len(ratios), interval=300))

Even at 75 % *you* can still guess that this is a boat in a harbour. That kind of guessing is exactly the skill MAE trains.
Let's ask a real pretrained MAE (ViT-Base) to fill in the gaps:

In [ ]:
from transformers import ViTMAEForPreTraining
mae = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base").eval().to(device)

MASK_RATIO = 0.75          # 🎤 live: try 0.5 or 0.9 and re-run this cell
mae.config.mask_ratio = MASK_RATIO
mae.vit.embeddings.config.mask_ratio = MASK_RATIO

names = ["trawler", "sailboat", "lifeboat", "fishing_boat"]
x = torch.stack([to_tensor(load_image(n, size=224)) for n in names]).to(device)
torch.manual_seed(0)
with torch.no_grad():
    out = mae(pixel_values=x)
recon = mae.unpatchify(out.logits)                                                 # (B, 3, 224, 224) predicted pixels
mask = out.mask[:, :, None].repeat(1, 1, 16 * 16 * 3)                              # (B, 196, 768) 1 = masked
mask = mae.unpatchify(mask)                                                        # (B, 3, 224, 224)
masked_input = x * (1 - mask) + mask * ((0.5 - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device))   # grey for hidden patches
pasted = x * (1 - mask) + recon * mask                                             # visible patches + reconstructed ones

fig, axes = plt.subplots(len(names), 4, figsize=(12, 12))
for r in range(len(names)):
    for c, (t, title) in enumerate([(x, "original"), (masked_input, f"input to the encoder ({MASK_RATIO:.0%} masked)"),
                                    (recon, "reconstruction"), (pasted, "reconstruction + visible patches")]):
        axes[r, c].imshow(to_image(t[r])); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

The reconstructions are blurry (MSE loss averages over all plausible answers), but the model clearly "knows" where the hull,
the water and the horizon go, *from a quarter of the patches and without ever seeing a label*.
**Result:** MAE features are excellent when **fine-tuned**, e.g. for global prediction tasks.

---
# 3 · DINO: self-distillation with no labels
**Key idea:** make the network **agree with itself** on different views of the same image.

**Views, again.** In notebook 02 we created **new views** of each digit (shifted by up to ±2 px) as data augmentation: the
image looks different, its meaning (the label) stays the same. DINO takes this one step further: it creates many views of
each photo (random crops, flips, colour changes) and **needs no label at all**. The only rule is: *all views of the same
image must give the same output*. A network can only satisfy that by learning what is in the image.

1. generate several random **crops** of the same image: 2 large *global* views + several small *local* views,
2. the **student** (a ViT, trained by gradient descent) sees **all** crops,
3. the **teacher** (same architecture) sees only the global views; its weights are an **exponential moving average (EMA)**
   of the student's, so it gets no gradient at all,
4. **loss:** cross-entropy $H(p_t, p_s)$ between the teacher's and the student's output distributions,
5. no labels required! (Centering and sharpening of the teacher outputs prevent the trivial solution "always output the same".)

In [ ]:
from torchvision import transforms
global_crop = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.4, 1.0)), transforms.RandomHorizontalFlip(),
                                  transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)])
local_crop = transforms.Compose([transforms.RandomResizedCrop(96, scale=(0.05, 0.4)), transforms.RandomHorizontalFlip(),
                                 transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)])
base = load_image("trawler")
torch.manual_seed(1)
fig, axes = plt.subplots(1, 7, figsize=(16, 3), gridspec_kw=dict(width_ratios=[1.5, 1.2, 1.2, .7, .7, .7, .7]))
axes[0].imshow(base); axes[0].set_title("original")
for k in range(2): axes[1 + k].imshow(global_crop(base)); axes[1 + k].set_title(f"global view {k + 1}\n→ teacher & student", color=NAVY, fontsize=9)
for k in range(4): axes[3 + k].imshow(local_crop(base)); axes[3 + k].set_title(f"local {k + 1}\n→ student", color=ORANGE, fontsize=9)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

The task for the student: *"from this small local crop, predict what the teacher says about the whole image"*.
To do that it has to learn what the object **is**, not what the pixels look like.

The teacher update is a single line, no backpropagation involved:

In [ ]:
@torch.no_grad()
def ema_update(teacher, student, momentum=0.996):
    for p_t, p_s in zip(teacher.parameters(), student.parameters()):
        p_t.mul_(momentum).add_((1 - momentum) * p_s)        # θ_teacher ← m·θ_teacher + (1 − m)·θ_student

### What does DINO learn? Look at the [CLS] attention
Below: the last-layer [CLS] attention of each of the 6 heads of a DINO ViT-S/8. Remember the noisy maps of the
supervised ViT in notebook 04? DINO was **never told what a boat is**, yet its heads segment the objects.

In [ ]:
from transformers import ViTModel
dino = ViTModel.from_pretrained("facebook/dino-vits8", attn_implementation="eager", add_pooling_layer=False).eval().to(device)

SIZE = 320                                                        # 320 / 8 = 40 × 40 patches
dino_names = ["trawler", "sailboat", "lifeboat", "ferry"]
xs = torch.stack([to_tensor(load_image(n, size=SIZE)) for n in dino_names]).to(device)
with torch.no_grad():
    att = dino(pixel_values=xs, output_attentions=True, interpolate_pos_encoding=True).attentions[-1]   # (B, 6, 1+1600, 1+1600)
cls_att = att[:, :, 0, 1:].reshape(len(dino_names), 6, SIZE // 8, SIZE // 8).cpu()

fig, axes = plt.subplots(len(dino_names), 7, figsize=(16, 9.6))
for r in range(len(dino_names)):
    axes[r, 0].imshow(to_image(xs[r]))
    for h in range(6):
        a = cls_att[r, h]; a = a / a.max()
        axes[r, h + 1].imshow(a, cmap="inferno")
        if r == 0: axes[r, h + 1].set_title(f"head {h + 1}")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("DINO ViT-S/8: last-layer [CLS] attention per head (no labels used in training!)", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

---
# 4 · DINOv2: dense features you can use directly
**DINOv2** (2023) scaled the recipe up (142 M curated images, ViT up to 1 B parameters) and added a masked-modelling loss.
Its **patch tokens** are so good that simple methods on top of *frozen* features work for retrieval, depth estimation and segmentation.

A popular way to *see* dense features: run **PCA** on all patch tokens and show the first three components as RGB.
Patches that the model considers similar get similar colours:

In [ ]:
from transformers import AutoModel
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").eval().to(device)

SIZE2, P2 = 448, 14                                              # 448 / 14 = 32 × 32 patch tokens
pca_names = ["trawler", "sailboat", "marina", "lifeboat", "fishing_boat", "ferry"]
x2 = torch.stack([to_tensor(load_image(n, size=SIZE2)) for n in pca_names]).to(device)
with torch.no_grad():
    tokens = dinov2(pixel_values=x2).last_hidden_state[:, 1:].cpu()   # (B, 1024, 384): drop [CLS]
G = SIZE2 // P2
feats = tokens.reshape(-1, tokens.shape[-1])                      # all patches of all images

# 1st principal component → foreground vs. background
from sklearn.decomposition import PCA
pc1 = PCA(n_components=1).fit_transform(feats.numpy())[:, 0]
border = np.zeros((G, G), bool); border[:2] = border[-2:] = border[:, :2] = border[:, -2:] = True
if pc1.reshape(-1, G, G)[:, border].mean() > 0: pc1 = -pc1         # the sign of a PC is arbitrary: make the image border (≈ background) negative
fg = pc1 > 0
# PCA with 3 components on the foreground patches only → RGB
pca3 = PCA(n_components=3).fit(feats[fg].numpy())
rgb = np.zeros((len(feats), 3)); rgb[fg] = pca3.transform(feats[fg].numpy())
rgb[fg] = (rgb[fg] - rgb[fg].min(0)) / (rgb[fg].max(0) - rgb[fg].min(0))

fig, axes = plt.subplots(3, len(pca_names), figsize=(16, 8))
for k in range(len(pca_names)):
    s = slice(k * G * G, (k + 1) * G * G)
    axes[0, k].imshow(to_image(x2[k]))
    axes[1, k].imshow(fg[s].reshape(G, G), cmap="gray")
    axes[2, k].imshow(rgb[s].reshape(G, G, 3))
for ax in axes.ravel(): ax.axis("off")
axes[0, 0].set_title("image", loc="left"); axes[1, 0].set_title("1st PC > 0: 'foreground'", loc="left"); axes[2, 0].set_title("PCA of foreground patches → RGB", loc="left")
plt.suptitle("DINOv2 patch features: segmentation with zero labels and zero training", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

With **no labels and no training at all**, a single PCA separates boats from water and sky. The same object parts get
the same colour across *different* images (hulls, masts, cabins, …), so the features are **semantic** and consistent between images.

---
# 5 · Benchmarking: evaluating frozen representations
**How do we know whether pretraining was good?**

The gold standard is to keep the backbone **frozen** (no gradient flows into it) and train only a tiny model on top:
* **linear probe:** a single linear layer on the features,
* **k-NN:** classify each test image by a majority vote of its nearest training images in feature space (no training at all).

*Why frozen?* Fine-tuning can compensate for bad representations. Frozen evaluation isolates the quality of the pretrained
features themselves. Common benchmarks: ImageNet top-1 (linear), COCO detection, ADE20k segmentation (linear).

We benchmark three frozen backbones on the maritime vessel dataset from notebook 03: six vessel types, only 50 labelled
photos per class.

In [ ]:
# @title 🗂️ The maritime dataset: 450 vessel photos from Open Images (run me) { display-mode: "form" }
# Human-verified image-level labels from Open Images V7 (annotations CC BY 4.0, photos from Flickr under CC BY 2.0).
# Photo authors are listed in assets/vessel_photo_credits.csv of the course repository.
import os, urllib.request
from concurrent.futures import ThreadPoolExecutor

VESSEL_IDS = {   # Open Images V7 image ids, 75 per class: the first 50 are the training set, the last 25 the test set
    "sailboat": """
    train/0017a5aca14e8a4f train/270f25208f428a20 train/01b0942287e46f96 train/042a9b5a1326e4bf
    train/4a6e88eca482b411 train/02dc87c4ffccca7e train/2f79b406c757c8ce train/289240aa3b8a1d77
    train/017d6e8b13d944c4 train/02452c3e8af9d4f4 train/0374707afaf5d93c train/15a3fc1ab5932a9f
    train/03725f2f9d997453 train/5cb74411e4e46105 train/0f8ab6b8fdcae79f train/4f601ccdde9eb719
    train/101d084a635efb76 train/000beb11fce5e3f1 train/341fa0c97a6a0c73 train/534223af95e2335b
    train/03d578aa62894dd3 train/02fe072f823a395f train/28846b6efb417e51 train/1d635f9654a4cd46
    train/19c7860b399a67d9 train/078f5ab0461695c8 train/515ff28a9f1214c8 train/09f08dcbc4fc970c
    train/191d9392f9a554ac train/5da8f60b54c25aed train/04a5935005756ce3 train/494267c94fbb7e22
    train/2f44e0335cfb5116 train/097a7d1e539776c2 train/0d2f380aba3fe3ae train/0201e168b1a350d4
    train/4e560f0a0211eab7 train/37955d9033610400 train/016e435568482491 train/19667b9e71c9fde4
    train/17141cd1d4223d10 train/0068ab21287bd575 train/474be7dcb750c0a6 train/2cd21e3ad1b530f4
    train/0112ee98d70904dc train/037303ad5d08ee8f train/30cf161b1f53c069 train/03ecb0ca96badd07
    train/4ca2a417d49b54bf train/01be612e619d1fab train/00268e680a609121 train/1b1eb10e184cadb1
    train/041ce864089465d1 train/010ebc3a9f513bef train/011acb31d18f4e19 train/07409b074d678f45
    train/11322b2db43e5d78 train/27bf2c8347c0756f train/2bdcec9243ec870d train/00a5053c35db851f
    train/114724f782fa1ac0 train/15f6cde66be01c59 train/054a4ee01ad343a4 train/468286170912d314
    train/234b0df2e4415e46 train/09f2edd4abfc4ca4 train/0a2d26bec1379d6d train/55de60f41eba09fe
    train/20045892a78d9d91 train/00d487e5e412be48 train/1f267a7867909f8e train/2bd3b48b53808dac
    train/20552329143c27b9 train/36f5f9763d76921a train/0e9d14c9c6ad80e8""",
    "speedboat": """
    test/7693cd525d384ed9 train/0ff28b7df418c29c train/447afa110efbc0f7 train/02f8c8ac608ae6a8
    train/01a0a526d767aaf1 train/00b934e52ca2240d test/386a81e65f7ba8a3 train/06c14a69d501708c
    train/82c156c2f63ed64a train/25be63ae33cd26eb test/55af54707631f77a train/dceaec9d14811a67
    train/06c2dd40e6ead0aa test/fc17acc08b558526 train/270372e9a68d7a3f train/ae7f44a5d7211736
    test/e311e37f66d84987 train/2bf611ad1fecd61d train/03e9ab6d9e190f18 train/50e0ed72f65c55bc
    test/d7f9c3791f6b3b84 train/0c6b77b7cfffbfcf test/bc1cab21b79505a6 train/a4e9e568527b578a
    train/0e1d00adc201c2ee train/a60d3a3127634a27 test/8877b4b24819da20 test/40d561a18ec315e5 test/c1f39ae6cfe350bd
    train/009135298b681ca3 train/001a0e2feef561c7 train/001a78066f572541 train/02c786e18af7ef0b
    train/085eb22a42f4500e test/55977d5de1b57012 train/3d46c82fdc0b26be train/067314c63fbdccb6
    train/2a36b9f16bf06db6 train/61bf9fa60b0a50ca train/032f4d2a18da0a5b test/f0cfb1215ae2b6e7
    train/3c222ac4a6257007 train/d95b966422e052e1 train/0110b0d005602f45 train/a6b0c488f90b0301
    train/0b25c4f261c87d26 test/72e65794f9f8b127 test/c2c95c571ef57115 train/007741a1be3f3280
    train/068a2062f112a16c train/10699dcd14dfac23 train/21154d871bdf54f4 test/d1d4028c10660c00
    train/308fffd977004718 test/9fe42bf848675308 test/d832f470f2ed2d4c test/bab7df5c8af2d8a3 train/15d98d56fdf37c5c
    train/092babacceaf5bb1 train/7f8f8404369adb38 train/d46ac4c76b9b5596 test/ed3a297b604eec23
    train/06ef034cee641dc9 train/0068af751c04c281 train/06749e42da77b4d3 train/1b50ee203ab8062e
    test/a37975f2d8eb65d2 test/f70169cae0b7e902 train/11b3a8f54fc2520f test/5772bbd05cb7aeba test/c52c840357f49938
    test/bf3201dd617f73cc test/8c18621a9e182de4 train/1240c0d08b65add4 train/016c4230676e2a15""",
    "ferry": """
    test/7a98c371f9d26bf2 test/1f0789232c23b9d2 test/1db6996b94c3af8d train/2020150fa5178ca1 test/6040600c1ec2e472
    test/7af863569ec4a543 train/19d96eee3abe4a84 test/271a63e888b97573 test/ac5baff7c4df4afa train/008be3073e8db77e
    test/f2a4fa73e6042b36 train/005b64d7a7dd7db0 train/3a36c4ecac8edc0c test/08f0e94891f99528 test/f604d0f25e7431a1
    test/7a6c885329d809f0 train/0debff76c6b5ff37 test/1e966f5df38fa8bd test/6b17965d00d4c158 test/b9e728ef283886e0
    test/c4cc47b37532f1b8 test/d2062c74661c9e1a train/49311edf36113265 test/e44496b9f3f3c0d5 train/00818942f783817e
    test/87b8418f60bf6c21 test/ddfce2d828bfb5e7 test/801f919125408088 test/bf0a5586b7b8bc6e test/5b5ffbce8bb67103
    test/5376fee27c87f8eb train/06fe127f335b5bb1 train/0a15968b8a6cb036 train/cbff57622ea2c658
    test/9eec387ad72111a2 train/02c1f4f557ee3d67 test/c36e48ca8d212f37 test/66e87d7c2381f2e0 test/9f4c9587dd89aba0
    test/154e7fac62e5a8d9 train/0204f979aef76229 test/6ca27b11eb161cbc train/13c835164a34991d
    train/00e541d948704a5c test/76725e536b6b9993 train/298c5839727ea7a6 train/268afe6a2d3a7552
    test/e27d79f519db6def test/a6f2076a108797cc test/05cfc55b16cc2b14 train/03c4c0c65e4de86d train/03b774fabee1be48
    train/4f67ac89f5549743 test/0b064592acaae176 train/5815969f9037efe8 train/936dfd8ec3ae0a15
    test/53537dfc83cd5cb4 train/00c9251579624f5d test/75009ae5d08aa0f9 train/009c8b4e2ade1548
    train/47d39754e0af8dd5 test/4d18cf7c46f4ff90 test/87d2058a8a36c914 train/3ac08b77af04dd1f
    train/c91f775749ac1a62 test/b07cf339fdba1b7a test/3ffdc7637447a73a train/01b913fad91691f3
    train/06a18b3568f2efd0 train/175314a3d631192c train/01c2cf0f69ed3b75 train/009bbf88c189af9c
    test/6fd7242ea3968eb3 test/64cf1f42b54d38fd train/b00138ac393a9ba4""",
    "cruise ship": """
    train/1194df37b9507d39 train/21c92b4caf9d650f train/2b7d1b6d8e3a8101 train/0abd34acde73436e
    train/154da29a2b355285 train/6ba557459e405558 train/9443bfdab08c84a5 train/166bb0b95f7f157f
    train/0ceb84244a75f993 train/3d1f93ea04f093a0 train/009bf85434579e4a train/c27f0eec4a63dde9
    train/da2c4aa1639939f8 train/0dfe47147739f9de train/5f4de6a26863b335 train/38c84339d2e7ab2a
    train/19893c7a52d7f065 train/2f2c711a7af0cce1 train/1884bfcce25186ee train/db2b1d1d881f2ab2
    train/0ebd7452b4c1de95 train/10fe56e0ee384e1c train/1e976dbdab3f1f89 train/167d922607de56d6
    train/47599faa7fe8815c train/fbca26e4db84a249 train/00baf4acd29fec99 train/193608f978720f25
    train/14480f52d3a0675d train/0f7567d4a625490c train/603ef2f86f78a33e train/1bb8980e279fe649
    train/87004156745333c6 train/5cea567b01ebce9c train/2695323175048c98 train/108a1677d9109f67
    train/c4bba4aaa5de1edd test/6887b48136d4397e train/06c4a91c722ba57d train/213ceb4d4d329a2e
    train/22217ff40561e30f train/31082c48aa5f0758 train/2b2e19a666a40f60 train/0226208568ff6550
    train/010077f5343afbcf train/104d9780b5021635 train/0181ba314fded795 train/0d999ae353fb5f10
    train/565eab9a3be85b1f train/0f52f4adc7b0d69a train/163c070c3ea395c2 train/4832b041da177924
    train/601f18a94a85734c train/225cdace04e3f927 train/11feb8084d60010c train/13848b2d40d3cf94
    train/306a731865a32388 test/708e654e90b179df train/2b3450813f2c22a5 train/04cd3b06f1c65499
    train/2fa1510cd301d5a8 train/11c97e6a23442521 train/01b3681ee8735786 train/03498b3ce8b7fea4
    train/eec75914faff4f42 train/0c6eb74b168da0c3 train/06b191ded4d6bc18 train/c058d825f61dbdd3
    train/3beafba73da26219 train/0e9364519fc8b490 train/0cc5f9f6c74a8a05 train/1f54da5e6e04c361
    train/fc734066ccb54806 train/3a7e08075667183c train/a40ec067dc2f8df2""",
    "cargo ship": """
    train/1a06ca6e11e53ebb train/6dc973bbd648cd37 test/d6015312d058c5e2 train/40c8e5a6b0c5359a
    train/d2ad679aab8780d6 train/4e912ab82b51efd9 test/627d76aba5ffbffd train/17b2fcaeeb09b271
    train/1631a3f452ab4822 train/07402aa5202e1cc8 test/3d8f90e8fa34060b train/b8d0db4ff71b361c
    train/1ef3cb1d0538e726 train/2c56e263d797e658 train/a62d7d77a34e9968 train/ab92430212fdc31d
    train/2bfdd42f520fe098 test/17ead6e57ffcabec train/1bb4428764418b3c train/131f58acd53530cc
    test/1164019caaac9ac7 train/00b6c0610dcad7d8 train/4ea783ca4357eeed train/01a49b6659c455c3
    train/0a1e2996b16ab13d test/aa23220c2e74390c test/8c1b6c2e0af0df67 train/0cf035cfdde6cbd4
    train/0779a3574a19db16 test/02914fda0e8fcf20 train/031ec9949229c38e train/1804bf398c7b74f0
    train/158646041834ff1b train/151437c63d5d056a train/be74dde43c8e5a09 test/d29139a884c79afa
    train/20fb3051a9455bba train/00bc4549af18ee8c test/c3def997d3332a63 train/00463e9506f3585b
    test/8dd9171a5b1aced2 train/3bce967f1430160b test/fa1b6aac88e3047b test/ecb46f1270bc9b10 train/29cf93f053578bc3
    test/d86fbe30ca29ddb6 train/1a0220308b5cacdd train/0c052c95c2d99bde train/60407e2ca7baa1e5
    train/080a7769c2e7e3a7 train/334dffc8e2c080f6 train/023e3cc3493f6763 train/d2ce0286bc5b3697
    test/a7419eb6679ff41b test/200cf602bb107ea5 train/4aa68d9932e820d1 test/5ad2697b3373b4b8 train/58ae9eb5bf30576d
    train/9b4fa7972b92e489 train/34e91523ebec1c2e train/75e4ad678705b775 train/d333f50ac0443bc5
    train/aebf96cf6280e9e9 train/06438a9a38882a34 train/36e99d49c5e30bc3 test/0b0e79034e64e08a
    train/0e03ad9a75b9934e train/b2c3a870c6faf8c6 test/25efb7b9f68f261b train/2075729770475a5c
    train/640f746ab8ff8721 train/13fc82e48ff64442 train/a12ec0b49f2a0fc0 train/de242501eee6e346
    train/0892f56b5bd4cdb8""",
    "fishing vessel": """
    train/00bffc9f1573d3b8 test/4cbbc0a34c6db938 test/b3782e5235c52733 train/022c67809939cc50
    train/030a01788b3ef318 test/26a88d4e0aff5018 train/00f7e5b6bd9c9f7d train/022a0b2bef00c583
    train/080f5c70438bf53a test/31fe61b00399f0f4 train/01b7c223365aadaf train/031f02e85ae63c7f
    test/1f20eead55d3dec6 train/15d5dc96d6b2f086 train/28de23353fb68135 train/03de0f0b21473f1f
    train/009612f7fc462117 train/0200e4c38e9a191d train/0427eb01462ed45f train/04ae0a59c3e8293f
    train/074ccca9a55a3bfb train/137d396060cdfcae train/00d102ecd20ff393 test/e6fdd541adcede78
    train/1ba7d0be09d14f00 train/00f61144be68b505 train/193ecdd081551a52 train/13d4897d7237294c
    train/0073d756da528f00 train/0339f3e454d4915c train/0034c88824c064a7 train/01092100b8cb5612
    train/1354c27c0b55f873 train/1561f68f83d3df2c train/a5b16ec7734944fc train/75bb854e92675085
    train/0103666f416102e1 train/01f86fd6f882feea train/0a0fc6fae177ead8 train/00b8cdfb794fd452
    train/27ec97a69bb66400 train/28d8c0a9e6b66461 train/07dd8cbd4bc66c99 train/0d69e56109699695
    test/c2679b0f70256712 train/02bca0d3d5b73d77 train/36928ced9ed8e738 train/03b980b6fed5f3e6
    train/095c2c6b32ffa8d1 train/289ee24cd60aa8a1 test/dea0a98898da0a60 train/14c00dd9b83773e9
    train/0d2f3c705bdd93c9 train/00bd0934443df5b2 train/0122a88bf2f1f3a7 train/c486238f81c3994a
    train/1eb83e23c291600a train/08011b1d8efa3a0e train/12e230b3ec3bf57f train/0083cbf8e02aa2f4
    train/10ca246878775d81 train/1b3c94bea20d7ce0 test/dbf5db7cc0fffe7d train/043fb92c60162d51
    train/03612b66016d51b1 train/0292342978c23c13 test/e6c9c7d9d98b1287 test/84f30b598db989f9 test/e6ef32fce751bb90
    train/01138d95e5ec7507 train/030e84ae66194a7d train/06416a56ee5697d8 train/026c1e50529f162a
    train/007154ad137d092f train/1170f52639ded833""",
}
VESSEL_CLASSES = list(VESSEL_IDS)
N_TRAIN_PER_CLASS = 50

def load_vessels(size=224, cache="vessels_224.pt"):
    """Download the photos once (≈ 90 MB), centre-crop them to squares, resize, and cache them as one uint8 tensor."""
    if os.path.exists(cache):
        return torch.load(cache)
    def fetch(key):
        split, image_id = key.split("/")
        for attempt in range(3):
            try:
                with urllib.request.urlopen(f"https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg", timeout=60) as r:
                    img = Image.open(io.BytesIO(r.read())).convert("RGB")
                s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
                return np.asarray(img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC))
            except Exception:
                time.sleep(1)
        return None                                           # give up on this photo, keep the rest
    items = [(c, i, key) for c, ids in enumerate(VESSEL_IDS.values()) for i, key in enumerate(ids.split())]
    with ThreadPoolExecutor(16) as ex:
        arrays = list(ex.map(lambda item: fetch(item[2]), items))
    keep = [k for k, a in enumerate(arrays) if a is not None]
    data = {"images": torch.tensor(np.stack([arrays[k] for k in keep])).permute(0, 3, 1, 2),     # (N, 3, size, size) uint8
            "labels": torch.tensor([items[k][0] for k in keep]),
            "is_train": torch.tensor([items[k][1] < N_TRAIN_PER_CLASS for k in keep])}
    if len(keep) < len(items): print(f"⚠️  {len(items) - len(keep)} photos could not be downloaded and are skipped")
    torch.save(data, cache)
    return data

In [ ]:
import torchvision
from torch.utils.data import TensorDataset, DataLoader
vessels = load_vessels()                                      # the same 450 photos as in notebook 03
imgs_train, labels_train = vessels["images"][vessels["is_train"]], vessels["labels"][vessels["is_train"]]
imgs_test, labels_test = vessels["images"][~vessels["is_train"]], vessels["labels"][~vessels["is_train"]]
N_CLASSES = len(VESSEL_CLASSES)

def prep(x_uint8, size=224):
    x = x_uint8.to(device).float() / 255
    if x.shape[-1] != size: x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device)

@torch.no_grad()
def extract(fn, imgs, batch_size=50):
    return torch.cat([fn(prep(imgs[i:i + batch_size])).float().cpu() for i in range(0, len(imgs), batch_size)])

resnet_sup = torchvision.models.resnet50(weights="IMAGENET1K_V2"); resnet_sup.fc = nn.Identity(); resnet_sup.eval().to(device)
resnet_rand = torchvision.models.resnet50(weights=None); resnet_rand.fc = nn.Identity(); resnet_rand.eval().to(device)

def dinov2_features(x):
    h = dinov2(pixel_values=x).last_hidden_state
    return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1)        # [CLS] ‖ mean of patch tokens (the DINOv2 linear-eval recipe)

backbones = {
    "ResNet-50, random init\n(no pretraining)": (resnet_rand, 25.6),
    "ResNet-50, supervised\n(ImageNet labels)": (resnet_sup, 25.6),
    "DINOv2 ViT-S/14\n(self-supervised)": (dinov2_features, 22.1),
}
features = {}
for name, (fn, _) in backbones.items():
    t0 = time.time(); features[name] = (extract(fn, imgs_train), extract(fn, imgs_test))
    print(f"{name.replace(chr(10), ' '):<55} dim {features[name][0].shape[1]}  ({time.time() - t0:.0f}s)")

### First, look inside: what did each backbone learn in its very first layer?
In notebook 02 we hand-crafted a 3 × 3 Sobel kernel, then watched gradient descent learn it from photo–edge-map pairs.
Here are first-layer filters before and after learning from data: the 16 strongest of the 64 7 × 7 RGB kernels of a random and
of a supervised ResNet-50, and for the self-supervised **DINO ViT-S/8** from Section 3 the 16 main components (PCA) of its 384
8 × 8 patch filters, the usual way to look at ViT patch filters. Each filter is stretched to the full colour range:

In [ ]:
# @title 🎨 Figure: first-layer filters of the three backbones { display-mode: "form" }
def strongest(W, n=16):                                      # the n filters with the largest weights
    return W[W.flatten(1).norm(dim=1).argsort(descending=True)[:n]]

def main_components(W, n=16):                                # PCA over all filters → the n main "filter patterns"
    X = W.flatten(1); X = X - X.mean(0)
    return torch.linalg.svd(X, full_matrices=False).Vh[:n].reshape(n, *W.shape[1:])

first_layers = {
    "ResNet-50, random init\n7 × 7 kernels, no pretraining": strongest(resnet_rand.conv1.weight.detach().cpu()),
    "ResNet-50, supervised\n7 × 7 kernels, ImageNet labels": strongest(resnet_sup.conv1.weight.detach().cpu()),
    "DINO ViT-S/8, self-supervised\n8 × 8 patch filters, no labels": main_components(dino.embeddings.patch_embeddings.projection.weight.detach().float().cpu()),
}
fig, axes = plt.subplots(3, 17, figsize=(16, 4.2), gridspec_kw=dict(width_ratios=[2.6] + [1] * 16))
for r, (name, W) in enumerate(first_layers.items()):
    lo, hi = W.amin(dim=(1, 2, 3), keepdim=True), W.amax(dim=(1, 2, 3), keepdim=True)
    W = (W - lo) / (hi - lo)
    axes[r, 0].axis("off")
    axes[r, 0].text(0.95, 0.5, name, ha="right", va="center", fontsize=9.5, color=NAVY, transform=axes[r, 0].transAxes)
    for k in range(16):
        axes[r, k + 1].imshow(W[k].permute(1, 2, 0), interpolation="nearest"); axes[r, k + 1].axis("off")
plt.suptitle("First-layer filters: noise without pretraining; edge, stripe and colour detectors after learning from data",
             color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

Without pretraining the filters are noise. The supervised ResNet learned **oriented edge and stripe detectors** (the Sobel
kernel's big brothers) and **colour blobs** from 1.28 M labelled images. The self-supervised ViT learned a similar vocabulary of
blobs, bars and colour contrasts **without a single label**. How useful are the features built on top of such filters?
Let's measure, with DINOv2, the larger successor of DINO.

### The k-NN classifier
For each test feature: cosine similarity to all training features → take the `k` most similar → majority vote of their labels. No training at all:

In [ ]:
def knn_accuracy(f_train, y_train, f_test, y_test, k=20):
    sim = F.normalize(f_test, dim=1) @ F.normalize(f_train, dim=1).T     # (n_test, n_train) cosine similarities
    nn_labels = y_train[sim.topk(k, dim=1).indices]                      # (n_test, k)
    pred = nn_labels.mode(dim=1).values                                  # majority vote
    return (pred == y_test).float().mean().item()

In [ ]:
def linear_probe(f_train, y_train, f_test, y_test, epochs=100):
    mu, sd = f_train.mean(0), f_train.std(0) + 1e-6                      # standardise features
    f_train, f_test = (f_train - mu) / sd, (f_test - mu) / sd
    probe = nn.Linear(f_train.shape[1], N_CLASSES); opt = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-3)
    for _ in range(epochs):
        for f, y in DataLoader(TensorDataset(f_train, y_train), batch_size=64, shuffle=True):
            loss = F.cross_entropy(probe(f), y); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return (probe(f_test).argmax(1) == y_test).float().mean().item()

torch.manual_seed(0)
results = {name: (knn_accuracy(ftr, labels_train, fte, labels_test, k=10), linear_probe(ftr, labels_train, fte, labels_test))
           for name, (ftr, fte) in features.items()}
for name, (knn, lin) in results.items(): print(f"{name.replace(chr(10), ' '):<55} k-NN {knn:.1%}   linear {lin:.1%}")

fig, ax = plt.subplots(figsize=(10, 4))
xpos = np.arange(len(results))
for k, (lbl, col) in enumerate([("k-NN (k = 10)", LAVENDER), ("linear probe", NAVY)]):
    vals = [v[k] for v in results.values()]
    b = ax.bar(xpos + (k - .5) * .36, vals, .34, color=col, label=lbl); ax.bar_label(b, labels=[f"{v:.0%}" for v in vals], fontsize=9)
ax.axhline(1 / N_CLASSES, color=GREY, ls=":", lw=1)
ax.set_xticks(xpos, [f"{n}\n{backbones[n][1]} M params" for n in results]); ax.set_ylim(0, 1.05)
ax.set_ylabel("test accuracy (6 vessel types)"); ax.legend(loc="upper left")
ax.set_title("Frozen backbones, 50 labelled photos per class: how good are the features?")
plt.tight_layout(); plt.show()

* A **random** network gives poor features, so *the architecture alone* is not the magic. Pretraining is.
* The **self-supervised** DINOv2 ViT-S (never saw a single label during pretraining, and slightly *smaller* than ResNet-50)
  gives the best features. Even the training-free k-NN classifier gets most vessel photos right.

---
# 6 · Applying VFMs: from linear heads to fine-tuning

| | trained parameters | when to use |
|---|---|---|
| **linear head** on frozen features | tiny ($d \times C$) | features already encode what you need; very few labels |
| **bigger head** (MLP, FPN, UPerNet, …) on frozen features | small | dense prediction, multi-scale fusion, non-linear tasks |
| **partial fine-tuning** (unfreeze the last $k$ blocks) | medium | domain shift, e.g. ImageNet → maritime/sonar/satellite |
| **full fine-tuning** | everything | lots of labels, large domain shift; needs a small learning rate |

*less adaptation ← linear · bigger head · partial · full fine-tuning → more adaptation*

**Dense prediction with a linear head:** keep the patch tokens instead of [CLS], reshape them to the spatial grid
($32\times32\times d$ above) and apply a **1 × 1 convolution**, which is the same linear layer applied to every patch:

In [ ]:
d, n_classes = 384, 5
patch_grid = tokens[:1].reshape(1, G, G, d).permute(0, 3, 1, 2)          # (1, 384, 32, 32)
conv1x1 = nn.Conv2d(d, n_classes, kernel_size=1)
linear = nn.Linear(d, n_classes); linear.weight.data = conv1x1.weight.data[:, :, 0, 0]; linear.bias.data = conv1x1.bias.data
same = torch.allclose(conv1x1(patch_grid), linear(patch_grid.permute(0, 2, 3, 1)).permute(0, 3, 1, 2), atol=1e-5)
print("per-pixel class map:", tuple(conv1x1(patch_grid).shape), "| 1×1 conv == per-patch linear layer:", same)

### The race: linear probe vs. fine-tuning with very few labels
Can we do better than the frozen linear probe by also training (part of) the backbone? We use our whole vessel training
set (**50 labelled photos per class**), compare four strategies on DINOv2 ViT-S, and record the test accuracy after every
epoch. *(Roughly a minute on a T4 GPU.)*

`set_trainable` freezes the whole backbone and then unfreezes only the last `k` Transformer blocks
(`k = 0`: frozen backbone; `k = 12`: full fine-tuning):

In [ ]:
def set_trainable(backbone, k):
    for p in backbone.parameters():
        p.requires_grad_(False)
    if k > 0:
        for block in backbone.encoder.layer[-k:]:
            for p in block.parameters(): p.requires_grad_(True)
        for p in backbone.layernorm.parameters(): p.requires_grad_(True)

In [ ]:
from torchvision import transforms

class Classifier(nn.Module):
    def __init__(self, backbone, head="linear", n_classes=N_CLASSES):
        super().__init__()
        self.backbone = backbone; d = backbone.config.hidden_size
        self.head = nn.Linear(2 * d, n_classes) if head == "linear" else nn.Sequential(
            nn.Linear(2 * d, 512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, n_classes))
    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1))

EPOCHS_FT = 8
amp = lambda: torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda")

def run_strategy(head, k_blocks, epochs=EPOCHS_FT, lr_head=1e-3, lr_backbone=2e-5):
    torch.manual_seed(0)
    model = Classifier(AutoModel.from_pretrained("facebook/dinov2-small"), head).to(device)
    set_trainable(model.backbone, k_blocks)
    params = [{"params": model.head.parameters(), "lr": lr_head}]
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    if bb: params.append({"params": bb, "lr": lr_backbone})
    opt = torch.optim.AdamW(params, weight_decay=0.05)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loader = DataLoader(TensorDataset(imgs_train, labels_train), batch_size=32, shuffle=True)
    flip = transforms.RandomHorizontalFlip()

    @torch.no_grad()
    def test_accuracy():
        model.eval(); correct = 0
        with amp():
            for i in range(0, len(imgs_test), 50):
                correct += (model(prep(imgs_test[i:i + 50])).argmax(1).cpu() == labels_test[i:i + 50]).sum().item()
        return correct / len(imgs_test)

    accs, t0 = [test_accuracy()], time.time()                     # epoch 0: before any training
    for epoch in range(epochs):
        model.train(); model.backbone.train(k_blocks > 0)
        for xb, yb in loader:
            with amp():
                loss = F.cross_entropy(model(prep(flip(xb))), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        accs.append(test_accuracy())
    return dict(accs=accs, params=n_trainable, secs=time.time() - t0)

strategies = {"linear head\n(frozen)": ("linear", 0), "MLP head\n(frozen)": ("mlp", 0),
              "fine-tune last 2\nblocks + head": ("linear", 2), "full fine-tuning": ("linear", 12)}
ft_results = {}
for name, (head, k) in strategies.items():
    ft_results[name] = r = run_strategy(head, k)
    print(f"{name.replace(chr(10), ' '):<34} acc {r['accs'][-1]:.1%}   trainable params {r['params']:>10,}   ({r['secs']:.0f}s)")

In [ ]:
# @title 🎞️ Animation: the race between the four adaptation strategies { display-mode: "form" }
names_ft = list(ft_results); colors_ft = [LAVENDER, BLUE, ORANGE, RED]
fig, (ax, axb) = plt.subplots(1, 2, figsize=(15, 4.6), dpi=85, gridspec_kw=dict(width_ratios=[1.5, 1]))
lines = [ax.plot([], [], "-o", color=c, lw=2.5, ms=4, label=n.replace("\n", " "))[0] for n, c in zip(names_ft, colors_ft)]
ax.axhline(1 / N_CLASSES, color=GREY, ls=":", lw=1)
ax.set(xlim=(0, EPOCHS_FT), ylim=(0, 1), xlabel="epoch", ylabel="test accuracy (150 photos)"); ax.legend(loc="lower right", fontsize=9)
bars = axb.barh(range(len(names_ft))[::-1], [0] * len(names_ft), color=colors_ft)
axb.set_yticks(range(len(names_ft))[::-1], names_ft, fontsize=9); axb.set_xlim(0, 1.18); axb.set_xlabel("test accuracy")
labels = [axb.text(0, y, "", va="center", fontsize=9) for y in range(len(names_ft))[::-1]]
title = fig.suptitle("", color=NAVY, fontweight="bold"); plt.tight_layout()

def update(e):
    for ln, n in zip(lines, names_ft): ln.set_data(range(e + 1), ft_results[n]["accs"][:e + 1])
    for b, t, n in zip(bars, labels, names_ft):
        a = ft_results[n]["accs"][e]; b.set_width(a)
        t.set_position((a + 0.01, b.get_y() + b.get_height() / 2)); t.set_text(f"{a:.0%}  ·  {ft_results[n]['params']:,} trained")
    title.set_text(f"epoch {e} / {EPOCHS_FT}")
    return []

play(animation.FuncAnimation(fig, update, frames=EPOCHS_FT + 1, interval=700))

Compare the curves with the number of trained parameters. With only **50 labelled photos per class** the four strategies end up
within a few test photos of each other (in our test run: frozen linear head 85 %, MLP head 81 %, last two blocks 83 %, full
fine-tuning 81 %), although full fine-tuning trains about **4,600 × more parameters**, takes several times longer and wobbles
on the way. With this few labels, the cheapest option, a frozen backbone with a linear head, is the one to beat.

Fine-tuning pays off when you have **more labels** or a **large domain shift**: sonar, infrared, underwater or aerial images look
nothing like the photos DINOv2 was pretrained on. Where you sit on this spectrum is a design decision: how many labels do you have,
how large is the domain shift, and how much compute can you spend at training and deployment time? That is a question you will
meet again in your own maritime projects.

### 🎤 Ask the audience
1. What changes with **ten times more labels** (500 photos per class)?

<details><summary><b>Answer</b></summary>

Fine-tuning profits most from more labels: partial and full fine-tuning typically pull ahead, while the frozen linear head improves only a little. Few labels → adapt little; many labels → adapt more.

</details>

2. What happens if we fine-tune the whole backbone with a **large learning rate** (`lr_backbone=1e-3` instead of `2e-5`)?

<details><summary><b>Answer</b></summary>

The first large updates overwrite the pretrained features (*catastrophic forgetting*). With only 50 labels per class there is not enough data to learn them back, and accuracy falls far below the frozen linear head.

</details>

3. Would a bigger backbone (`facebook/dinov2-base`, 86 M parameters) help?

<details><summary><b>Answer</b></summary>

Usually yes, for every strategy: better features, higher accuracy. The price is about 4× the compute, for training and for every prediction at deployment time.

</details>

---
# 7 · Closing: "how foundational is it?"
The term *foundation model* has no single agreed definition. Models differ not in **kind** but in **degree**. Ask these
questions about any model you meet:

| | ResNet-50 (ImageNet, 2015) | MAE ViT-B (2021) | DINOv2 ViT-S (2023) |
|---|---|---|---|
| pretraining data | 1.28 M images, **labelled** | 1.28 M images, **no labels** | 142 M curated images, **no labels** |
| how many downstream tasks? | classification; detection/segmentation after fine-tuning | strong after fine-tuning | classification, retrieval, segmentation, depth, … with frozen features |
| adaptation needed | fine-tuning per task | fine-tuning | often just a linear head |
| transfer to unseen domains | moderate | good | very good (still: test it on *your* data!) |

**For maritime vision we are at the frontier.** The core concepts from today (pretraining, frozen evaluation, domain
adaptation) are the tools we will apply throughout this course.

---
# Summary
1. **Self-supervised learning** creates labels from the data itself and makes pretraining on unlabelled images possible.
2. **MAE:** mask 75 % of the patches, reconstruct the pixels; the encoder only sees the visible ones.
3. **DINO:** a student matches an EMA teacher across different crops; its attention segments objects without labels.
4. **DINOv2** patch features are semantic enough to segment boats with a single PCA.
5. **Frozen evaluation** (linear probe, k-NN) measures representation quality; self-supervised features can beat supervised ones.
6. **Adaptation spectrum:** linear head → bigger head → partial → full fine-tuning; choose by data size and domain shift.

> **Take-home message:** the field has moved from *training from scratch* to *adapting pretrained representations*.
> Understanding what a model learned during pretraining is as important as understanding its architecture.

**Next session:** Scientific writing for ML papers: how to structure a methods section, and how to cite and ablate VFMs correctly.